# Descarga de sentencias del STJ Jalisco

Pasos de `descargar.py` desde un notebook (detalle en `README.md` de esta carpeta). El modo **navegador** abre una
ventana visible de Chromium: una persona resuelve el reCAPTCHA y la descarga continúa con esa sesión; si el portal
vuelve a pedirlo, el script espera a la persona. El script nunca resuelve ni evita el reCAPTCHA.

**1. Requisitos** (una sola vez).

In [ ]:
%pip install -q requests playwright
!playwright install chromium

**2. Parámetros.** Carpeta de destino, materias (`penal`, `civil`, `familiar`, `mercantil` o `todas`), delito opcional y tope por tipo.

In [ ]:
DESTINO = "../data/raw/nuevas"   # o cualquier carpeta
MATERIAS = "penal"                # "todas" para todas las materias
DELITO = ""                       # texto a buscar en el delito o acción; vacío = todos
MAX_POR_TIPO = 30                 # vacío = sin tope

**3. Catálogo de asuntos** (opcional): la lista de lo publicado, sin descargar PDF (`catalogo_asuntos.csv`).

In [ ]:
!python descargar.py --destino "$DESTINO" --materias "$MATERIAS" --solo-catalogo --modo navegador

**4. Descarga.** Se puede interrumpir y volver a ejecutar: lo completo no se repite.

In [ ]:
opciones = f'--destino "{DESTINO}" --materias "{MATERIAS}" --modo navegador'
opciones += f' --delito "{DELITO}"' if DELITO else ""
opciones += f" --max-por-tipo {MAX_POR_TIPO}" if MAX_POR_TIPO else ""
!python descargar.py {opciones}

**5. Registro** de la descarga.

In [ ]:
import pandas as pd
from pathlib import Path

registro = pd.read_csv(Path(DESTINO) / "registro_descargas.csv")
registro.groupby(["materia", "estado"]).size().unstack(fill_value=0)

**6. Incorporar al corpus** (desde la raíz del repositorio):

```bash
python -m copiloto_juridico.cli construir-corpus --pdfs data/raw/nuevas
python -m copiloto_juridico.cli indexar
```